In [ ]:
import os
import re
from google.colab import drive
from pathlib import Path

print("⏳ TAHAP 1: Menyiapkan ruang kerja Tashkeela...")

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/BiLSTM-CRF')
path_raw = PROJECT_ROOT / 'data' / 'raw'
path_processed = PROJECT_ROOT / 'data' / 'raw'

path_processed.mkdir(parents=True, exist_ok=True)
print("✅ Konfigurasi folder kerja selesai!")
print("Sumber korpus :", path_raw)
print("Output sampel :", path_processed / 'tashkeela_sampled.csv')

# Pembersihan awal Tashkeela mengikuti rancangan penelitian:
# mempertahankan aksara Arab, spasi, dan tanda baca yang digunakan.
def bersihkan_dan_potong_kalimat(teks_mentah):
    teks_bersih = re.sub(r'[^\u0600-\u06FF\s\.\(\)\[\]،؛؟]', '', teks_mentah)
    teks_bersih = re.sub(r'\s+', ' ', teks_bersih).strip()
    kalimat_list = re.split(r'[\.؛؟]', teks_bersih)
    return [k.strip() + "." for k in kalimat_list if len(k.strip()) > 20]

def hilangkan_harakat(teks_berharakat):
    pola_harakat = re.compile(r'[\u064B-\u065F\u0670]')
    return re.sub(pola_harakat, '', teks_berharakat)

print("✅ Fungsi pembersihan siap digunakan.")


In [ ]:
import os
import pandas as pd
from tqdm.notebook import tqdm

print("🚀 TAHAP 2: Memulai Ekstraksi Tashkeela dengan Filter Karakter & Reset Index...\n")

# Mengambil kembali fungsi pembersih dari Tahap 1 dengan pembaruan filter panjang kalimat
def bersihkan_dan_potong_kalimat_final(teks_mentah):
    teks_bersih = re.sub(r'[^\u0600-\u06FF\s\.\(\)\[\]،؛؟]', '', teks_mentah)
    teks_bersih = re.sub(r'\s+', ' ', teks_bersih).strip()
    kalimat_list = re.split(r'[\.؛؟]', teks_bersih)

    # SARAN TERAPKAN: Membatasi panjang kalimat antara 20 sampai 250 karakter
    # Kalimat di bawah 20 terlalu pendek, di atas 250 tidak digunakan pada tahap pembentukan sampel
    return [k.strip() + "." for k in kalimat_list if 20 <= len(k.strip()) <= 250]

daftar_kalimat_harakat = []
daftar_kalimat_gundul = []
semua_file_txt = []

# 1. MENCARI FILE DI SELURUH SUB-FOLDER
# Membaca seluruh 309 berkas Tashkeela dari subfolder train/val/test sebagai satu korpus.
for root, dirs, files in os.walk(str(path_raw)):
    for file in files:
        if file.endswith('.txt'):
            semua_file_txt.append(os.path.join(root, file))

print(f"📂 Berhasil menemukan {len(semua_file_txt)} file .txt.")

if len(semua_file_txt) == 0:
    print("❌ ERROR: File tidak ditemukan. Pastikan path_raw sudah benar.")
else:
    # 2. PROSES EKSTRAKSI
    for path_file in tqdm(semua_file_txt, desc="Memproses"):
        try:
            with open(path_file, 'r', encoding='utf-8') as file:
                teks_mentah = file.read()
                potongan_kalimat = bersihkan_dan_potong_kalimat_final(teks_mentah)

                for kalimat in potongan_kalimat:
                    if kalimat.count(" ") >= 2:
                        teks_gundul = hilangkan_harakat(kalimat)
                        daftar_kalimat_harakat.append(kalimat)
                        daftar_kalimat_gundul.append(teks_gundul)
        except:
            continue

    # 3. PEMBERSIHAN DUPLIKAT
    df_tashkeela = pd.DataFrame({
        'teks_gundul': daftar_kalimat_gundul,
        'teks_harakat': daftar_kalimat_harakat
    }).drop_duplicates(subset=['teks_gundul'])

    # 4. SAMPLING & RESET INDEX
    JUMLAH_TARGET = 7000
    if len(df_tashkeela) > JUMLAH_TARGET:
        # Mengambil 7000 baris dan merapikan nomor urut baris (Reset Index)
        df_sampled = df_tashkeela.sample(n=JUMLAH_TARGET, random_state=42).reset_index(drop=True)
        print(f"⚖️ Berhasil sampling {JUMLAH_TARGET} baris dengan indeks yang rapi.")
    else:
        df_sampled = df_tashkeela.reset_index(drop=True)

    # 5. PENYIMPANAN FINAL
    path_simpan = path_processed / 'tashkeela_sampled.csv'
    df_sampled.to_csv(path_simpan, index=False, encoding='utf-8-sig')

    print(f"\n🎉 ALHAMDULILLAH! TAHAP 2 SELESAI.")
    print(f"📊 Total Data Tashkeela Siap Pakai: {len(df_sampled)} baris.")
    print(f"💾 Lokasi: {path_simpan}")

    print("\n=== CONTOH DATA FINAL ===")
    print(df_sampled.head(3))